# Infraestrutura e QoS do dump Argus

## Escopo, janela e limites

Cada leitura usa a janela UTC explícita `[start, end)` e parâmetros vinculados.
Os loaders limitam cada DataFrame a no máximo 100.000 linhas; `prom_point` nunca
é lida sem antes selecionar uma seção e no máximo 20 séries.

In [ ]:
from dump_explorer.charts import event_overlay_timeline, qos_small_multiples
from dump_explorer.config import load_settings
from dump_explorer.database import create_db_engine
from dump_explorer.qos import (
    load_apollo,
    load_cloudarmor,
    load_newrelic,
    load_prometheus,
    load_rpaas,
    load_tsuru_events,
)

# derive a safe UTC epoch-ms window from signal_bucket observed bounds
from dump_explorer.quality import load_source_coverage
import pandas as pd

settings = load_settings()
engine = create_db_engine(settings)

s = load_source_coverage(engine)           # rápido; retorna first_ts/last_ts em UTC
if s.empty or s['first_ts'].dropna().empty:
    raise RuntimeError("source_coverage vazio — verifique conexão/importe o dump")
start_ms = int(s['first_ts'].dropna().min().value // 10**6)
end_ms = int(s['last_ts'].dropna().max().value // 10**6) + 1
print("start_ms=", start_ms, "end_ms=", end_ms)
print("start (UTC) =", pd.to_datetime(start_ms, unit='ms', utc=True))
print("end   (UTC) =", pd.to_datetime(end_ms-1, unit='ms', utc=True))

## 1. New Relic e Apollo

Apdex tem direção `lower_is_worse`. Em Apollo, `request_count` é contagem por
linha: `rpm` usa a duração real de cada janela, que não é fixa.

In [ ]:
newrelic_apps = []  # opcional: nomes de aplicações
apollo_operations = []  # opcional: operações GraphQL

# Aviso se listas vazias
if not newrelic_apps:
    print("⚠️  newrelic_apps está vazia. Descomente e adicione nomes de aplicações para visualizar dados.")
else:
    newrelic = load_newrelic(engine, newrelic_apps, start_ms, end_ms)
    if not newrelic.empty:
        try:
            qos_small_multiples(newrelic).show()
        except Exception as e:
            print(f"❌ Erro ao renderizar New Relic: {e}")
    else:
        print("⚠️  Nenhum dado encontrado para as aplicações New Relic especificadas.")

if not apollo_operations:
    print("⚠️  apollo_operations está vazia. Descomente e adicione operações GraphQL para visualizar dados.")
else:
    apollo = load_apollo(engine, apollo_operations, start_ms, end_ms)
    if not apollo.empty:
        try:
            qos_small_multiples(apollo).show()
        except Exception as e:
            print(f"❌ Erro ao renderizar Apollo: {e}")
    else:
        print("⚠️  Nenhum dado encontrado para as operações Apollo especificadas.")

## 2. RPaaS e Cloud Armor

RPaaS é amostrado a **1%**: as contagens são amostras, não tráfego absoluto, e
as taxas (`5xx/total`) são mais comparáveis. A taxa 4xx é contextual; somente
os vhosts de playback recebem `playback_signal=True`. Um aparente pico de
memória em Prometheus pode ser o limite do **cgroup** (por exemplo,
`gg-gateway-rpaas-be`), e não uso real de memória.

In [ ]:
rpaas_regions = []  # opcional: ["br1"]
rpaas_vhosts = []  # opcional: ["playback.video.globo.com"]
cloudarmor_apps = []  # opcional: aplicações protegidas

# RPaaS
if not rpaas_regions and not rpaas_vhosts:
    print("⚠️  rpaas_regions e rpaas_vhosts vazios. Descomente e adicione filtros para visualizar dados.")
else:
    rpaas = load_rpaas(engine, rpaas_regions, rpaas_vhosts, start_ms, end_ms)
    if not rpaas.empty:
        try:
            qos_small_multiples(rpaas).show()
        except Exception as e:
            print(f"❌ Erro ao renderizar RPaaS: {e}")
    else:
        print("⚠️  Nenhum dado encontrado para RPaaS com os filtros especificados.")

# Cloud Armor
if not cloudarmor_apps:
    print("⚠️  cloudarmor_apps está vazia. Descomente e adicione aplicações para visualizar dados.")
else:
    cloudarmor = load_cloudarmor(engine, cloudarmor_apps, start_ms, end_ms)
    if not cloudarmor.empty:
        try:
            qos_small_multiples(cloudarmor).show()
        except Exception as e:
            print(f"❌ Erro ao renderizar Cloud Armor: {e}")
    else:
        print("⚠️  Nenhum dado encontrado para Cloud Armor com os filtros especificados.")

## 3. Seções selecionadas do Prometheus

Escolha explicitamente as seções e, se necessário, serviços. O limite de 20
séries é aplicado antes do join com `prom_point`; não use a seção para inferir
semântica sem verificar o painel e os labels.

In [ ]:
prometheus_sections = ["RPAASV2 CPU"]  # selecione seções de interesse
prometheus_services = []  # opcional: services
technical_for_events = None

for section in prometheus_sections:
    prom_section = load_prometheus(
        engine, section, prometheus_services, start_ms, end_ms, limit_series=20
    )
    qos_small_multiples(prom_section).show()
    if technical_for_events is None:
        technical_for_events = prom_section

## 4. Eventos Tsuru sobrepostos

`autoscale.run` significa reaplicação de regra (`rule_reapplication`), não
prova de escala por demanda. A sobreposição temporal é contexto para
investigação, não evidência de causalidade. Campos sanitizados como owner e
error permanecem nulos.

In [ ]:
tsuru_targets = []  # opcional: nomes de apps/pools
tsuru_events = load_tsuru_events(engine, tsuru_targets, start_ms, end_ms)
if technical_for_events is not None and not tsuru_events.empty:
    try:
        event_overlay_timeline(technical_for_events, tsuru_events).show()
    except Exception as e:
        print(f"❌ Erro ao renderizar timeline de eventos Tsuru: {e}")
if not tsuru_events.empty:
    display(tsuru_events.head())
else:
    print("⚠️  Nenhum evento Tsuru encontrado no intervalo.")